In [1]:
import dataclasses
from functools import partial
from typing import Any, Callable
import os

os.environ["LIBTPU_INIT_ARGS"] = " ".join([
  #"--deepsea_chip_config_name=megachip_tccontrol",
  #"--deepsea_version=ghostlite",
  #"--2a886c8_chip_config_name=megachip_tccontrol"
  #"--2a886c8_version=cc8675309"
  #"--xla_tpu_enable_offloading_scatter_to_sparsecore=true"
  #"--xla_tpu_use_tc_device_shape_on_sc=true"
])

import jax
import jax.numpy as jnp
from jax import lax
from jax import random
from jax.sharding import PartitionSpec as P, auto_axes
import jax.experimental.pallas as pl
import jax.experimental.pallas.tpu as pltpu
import numpy as np
jax.devices()

[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0),
 TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0),
 TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0),
 TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0)]

# test ragged_all_to_all

In [3]:
DEVICE_NUM = jax.device_count()

n, k = 4 * (1024 ** 3) // 8192 // 2 * DEVICE_NUM, 8192
print(f"{(n, k)=}")

mesh = jax.make_mesh((DEVICE_NUM,), ("x",), axis_types=jax.sharding.AxisType.Explicit)
print(mesh)
jax.sharding.set_mesh(mesh)

x = jax.jit(lambda: jnp.ones((n, 8, k // 8), "bfloat16"), out_shardings=P("x", None, None))()

@jax.jit
def transfer(x):
  @partial(jax.shard_map, out_specs=P("x", None))
  def fn(x):
    peer_num = jax.lax.axis_size("x")
    chunk_shape = x.shape[0] // peer_num
    print(f"chunk_shape = {chunk_shape}")
    input_offsets = chunk_shape * jnp.arange(peer_num)
    output_offsets = chunk_shape * jnp.arange(peer_num)
    send_sizes = chunk_shape * jnp.ones(peer_num, "int32")
    recv_sizes = chunk_shape * jnp.ones(peer_num, "int32")
    output = jnp.zeros_like(x)

    #return jnp.sum(jax.lax.all_to_all(output, "x", 0, 0, tiled=True), 0)[None, :]

    #return jnp.sum(output, 0)[None, :]

    out = jax.lax.ragged_all_to_all(x, output, input_offsets, send_sizes, output_offsets, recv_sizes, axis_name="x")
    return jnp.sum(out, 0)[None, ...]

  return fn(x)

(n, k)=(1048576, 8192)
Mesh('x': 4, axis_types=(Explicit,))


In [8]:
x.shape

(1048576, 8, 1024)

In [7]:
with jax.profiler.trace("/tmp/ra2a"):
  for _ in range(3):
    y = jax.block_until_ready(transfer(x))

In [6]:
y.shape

(4, 8, 1024)

# non-aligned copy detour

In [ ]:
def place_result(x_ref, out_ref, offset_ref, size_ref, out_alias_ref, *, block_size: int):
  #del out_alias

  offset, size = offset_ref[0], size_ref[0]
  block_offset = offset // block_size
  shift = lax.rem(offset, block_size)

  def kernel(src_ref, dst_ref):
    i = pl.program_id(0)
    up_phase = i % 2 == 0
    val = src_ref[...]
    val = pltpu.roll(src_ref[...], jnp.where(up_phase, shift, -shift), 0)
    iota = lax.broadcasted_iota(jnp.int32, val.shape, 0)
    mask_up = iota >= (iota.shape[0] - shift)
    mask_down = iota < (iota.shape[0] - shift)
    #mask = jnp.where(i % 2 == 0, mask_up, mask_down)
    mask = (up_phase * mask_up) | (~up_phase * mask_down)
    #pl.debug_print("mask = {}\nval = {}\n---------------", mask[:, 0] * 1, val[:, 0])
    dst_ref[...] = jnp.where(mask, val, dst_ref[...])

  in_spec = pl.BlockSpec((block_size,) + x_ref.shape[1:], lambda i: (i // 2, 0))
  out_spec = pl.BlockSpec((block_size,) + out_ref.shape[1:], lambda i: ((block_offset + i + 1) // 2, 0))

  pltpu.sync_copy(x_ref.at[pl.ds(offset, size), :], out_ref.at[pl.ds(0, size), :])

  #pl.debug_print("grid = {}", pl.cdiv(size, block_size) * 2)

  #pltpu.emit_pipeline(
  #  kernel,
  #  grid=(pl.cdiv(size, block_size) * 2,),
  #  in_specs=[in_spec, out_spec],
  #  should_accumulate_out=True,
  ##)(x_ref, out_ref)
  #)(x_ref, out_alias_ref)


def shifted_copy(x, out, offset, size):
  return pl.pallas_call(
    partial(place_result, block_size=8),
    out_shape=out,
    in_specs=[pl.BlockSpec(memory_space=pl.MemorySpace.ANY)] * 2 + [pl.BlockSpec(memory_space=pltpu.MemorySpace.SMEM)] * 2,
    out_specs=pl.BlockSpec(memory_space=pl.MemorySpace.ANY),
    input_output_aliases={1: 0},
    interpret=False,
  )(x, out, jnp.atleast_1d(offset), jnp.atleast_1d(size))

In [ ]:
x = jax.lax.broadcasted_iota(jnp.int32, (128, 128), 0)
o = jnp.ones(x.shape, "int32")

In [ ]:
out = jax.jit(shifted_copy)(x, o, 13, 64)
out

# ra2a

In [2]:
AsyncCopyDescriptor = Any

@dataclasses.dataclass
class RDMACopy:
  copy: AsyncCopyDescriptor | None
  start: Callable[[], None]
  wait: Callable[[], None]

In [4]:
def _ra2a_kernel_async(src_ref, out_ref, input_offsets, send_sizes, output_offsets, recv_sizes, dst_ref, sems, *, axis_name, start: bool = True):
  del out_ref  # aliased in dst_ref
  idx, n_devices = jax.lax.axis_index(axis_name), jax.lax.axis_size(axis_name)

  def make_dma(id):
    sem_id = lax.rem(idx + idx, n_devices)
    src = src_ref.at[pl.ds(input_offsets[id], send_sizes[id]), ...]
    dst = dst_ref.at[pl.ds(output_offsets[id], send_sizes[id]), ...]
    copy = pltpu.make_async_copy(src, dst, sems.at[0, sem_id, 1])
    return RDMACopy(copy, copy.start, copy.wait)

  dma_copy = make_dma(idx)
  if start:
    dma_copy.start()

  def make_rdma(other_id, send: bool = True):
    src_id, dst_id = (idx, other_id) if send else (other_id, idx)
    size = lax.select(idx == src_id, send_sizes[dst_id], recv_sizes[src_id])
    src = src_ref.at[pl.ds(input_offsets[dst_id], size), ...]
    dst = dst_ref.at[pl.ds(output_offsets[dst_id], size), ...]
    sem_id, direction_id = lax.rem(idx + other_id, n_devices), (src_id > dst_id).astype(jnp.int32)
    send_sem = sems.at[sem_id, direction_id, 0]
    recv_sem = sems.at[sem_id, direction_id, 1]
    copy = pltpu.make_async_remote_copy(src, dst, send_sem, recv_sem, device_id=dst_id)
    start_fn = copy.start if send else (lambda: None)
    wait_fn = copy.wait_send if send else copy.wait_recv
    return RDMACopy(copy, start_fn, wait_fn)

  send_rdmas, recv_rdmas = [], []
  for i in range(1, n_devices):
    other_id = jax.lax.rem(idx + i, n_devices)
    send_rdmas.append(make_rdma(other_id, send=True))
    recv_rdmas.append(make_rdma(other_id, send=False))

  if start:
    [rdma.start() for rdma in (send_rdmas + recv_rdmas)]
  else:
    [rdma.wait() for rdma in (send_rdmas + recv_rdmas)]
    dma_copy.wait()

def make_ra2a(axis_name: str = "x"):

  def start(src, output, input_offsets, send_sizes, output_offsets, recv_sizes):
    n_devices = jax.lax.axis_size(axis_name)

    def ra2a_kernel_start(src_ref, out_ref, input_offsets, send_sizes, output_offsets, recv_sizes, dst_ref, sems):
      kws = dict(axis_name=axis_name, start=True)
      return _ra2a_kernel_async(
          src_ref, out_ref, input_offsets, send_sizes, output_offsets, recv_sizes, dst_ref, sems, **kws
      )

    sems_spec = pltpu.SemaphoreType.DMA((n_devices, 2, 2))
    out, sems = pl.pallas_call(
      ra2a_kernel_start,
      out_shape=[output, sems_spec],
      in_specs=2 * [pl.BlockSpec(memory_space=pltpu.ANY)] + 4 * [pl.BlockSpec(memory_space=pltpu.SMEM)],
      out_specs=[pl.BlockSpec(memory_space=pltpu.ANY), pl.BlockSpec(memory_space=pltpu.SEMAPHORE)],
      input_output_aliases={1: 0},
      interpret=False,
    )(src, output, input_offsets, send_sizes, output_offsets, recv_sizes)
    return out, sems

  def wait(src, output, input_offsets, send_sizes, output_offsets, recv_sizes, future):
    n_devices = jax.lax.axis_size(axis_name)
    sems = future

    def ra2a_kernel_wait(src_ref, out_ref, input_offsets, send_sizes, output_offsets, recv_sizes, sems, dst_ref):
      kws = dict(axis_name=axis_name, start=False)
      return _ra2a_kernel_async(
          src_ref, out_ref, input_offsets, send_sizes, output_offsets, recv_sizes, dst_ref, sems, **kws
      )

    # sems_spec = pltpu.SemaphoreType.DMA((n_devices, 2, 2))
    out = pl.pallas_call(
      ra2a_kernel_wait,
      out_shape=output,
      in_specs=2 * [pl.BlockSpec(memory_space=pltpu.ANY)] + 4 * [pl.BlockSpec(memory_space=pltpu.SMEM)] + [pl.BlockSpec(memory_space=pltpu.SEMAPHORE)],
      out_specs=pl.BlockSpec(memory_space=pltpu.ANY),
      input_output_aliases={1: 0},
      interpret=False,
    )(src, output, input_offsets, send_sizes, output_offsets, recv_sizes, sems)
    return out
    
  return start, wait

In [3]:
def ra2a_kernel(src_ref, out_ref, input_offsets, send_sizes, output_offsets, recv_sizes, dst_ref, sems, *, axis_name):
  del out_ref  # aliased in dst_ref
  idx, n_devices = jax.lax.axis_index(axis_name), jax.lax.axis_size(axis_name)

  def make_dma(id):
    sem_id = lax.rem(idx + idx, n_devices)
    #src = src_ref.at[pl.ds(input_offsets[id], send_sizes[id]), ...]
    #dst = dst_ref.at[pl.ds(output_offsets[id], send_sizes[id]), ...]
    src = src_ref.at[pl.ds(input_offsets[id], send_sizes[id]), ...]
    dst = dst_ref.at[pl.ds(output_offsets[id], send_sizes[id]), ...]
    copy = pltpu.make_async_copy(src, dst, sems.at[0, sem_id, 1])
    return RDMACopy(copy, copy.start, copy.wait)

  dma_copy = make_dma(idx)
  dma_copy.start()

  def make_rdma(other_id, send: bool = True):
    src_id, dst_id = (idx, other_id) if send else (other_id, idx)
    size = lax.select(idx == src_id, send_sizes[dst_id], recv_sizes[src_id])
    src = src_ref.at[pl.ds(input_offsets[dst_id], size), ...]
    dst = dst_ref.at[pl.ds(output_offsets[dst_id], size), ...]
    sem_id, direction_id = lax.rem(idx + other_id, n_devices), (src_id > dst_id).astype(jnp.int32)
    send_sem = sems.at[sem_id, direction_id, 0]
    recv_sem = sems.at[sem_id, direction_id, 1]
    copy = pltpu.make_async_remote_copy(src, dst, send_sem, recv_sem, device_id=dst_id)
    start_fn = copy.start if send else (lambda: None)
    wait_fn = copy.wait_send if send else copy.wait_recv
    return RDMACopy(copy, start_fn, wait_fn)

  send_rdmas, recv_rdmas = [], []
  for i in range(1, n_devices):
    other_id = jax.lax.rem(idx + i, n_devices)
    send_rdmas.append(make_rdma(other_id, send=True))
    recv_rdmas.append(make_rdma(other_id, send=False))
  [rdma.start() for rdma in send_rdmas]
  [rdma.wait() for rdma in (send_rdmas + recv_rdmas)]
  dma_copy.wait()

@partial(jax.jit, static_argnames=("axis_name",))
def ra2a(src, output, input_offsets, send_sizes, output_offsets, recv_sizes, *, axis_name: str = "x"):
  n_devices = jax.lax.axis_size(axis_name)
  out = pl.pallas_call(
    partial(ra2a_kernel, axis_name=axis_name),
    out_shape=output,
    in_specs=2 * [pl.BlockSpec(memory_space=pltpu.ANY)] + 4 * [pl.BlockSpec(memory_space=pltpu.SMEM)],
    out_specs=pl.BlockSpec(memory_space=pltpu.ANY),
    scratch_shapes=[pltpu.SemaphoreType.DMA((n_devices, 2, 2))],
    input_output_aliases={1: 0},
    interpret=False,
  )(src, output, input_offsets, send_sizes, output_offsets, recv_sizes)
  return out


In [4]:
DEVICE_NUM = jax.device_count()

n, k = 4 * (1024 ** 3) // 8192 // 2 * DEVICE_NUM, 8192
mesh = jax.make_mesh((DEVICE_NUM,), ("x",), axis_types=jax.sharding.AxisType.Explicit)
print(mesh)
jax.sharding.set_mesh(mesh)

Mesh('x': 4, axis_types=(Explicit,))


In [28]:
def ra2a_kernel(src_ref, out_ref, input_offsets, send_sizes, output_offsets, recv_sizes, dst_ref, sems, *, axis_name):
  del out_ref  # aliased in dst_ref
  idx, n_devices = jax.lax.axis_index(axis_name), jax.lax.axis_size(axis_name)

  def make_dma(id):
    sem_id = lax.rem(idx + idx, n_devices)
    #src = src_ref.at[pl.ds(input_offsets[id], send_sizes[id]), ...]
    #src = src_ref.reshape((src_ref.shape[0], src_ref.shape[1] * src_ref.shape[2])).at[pl.ds(input_offsets[id], send_sizes[id]), ...]
    src = src_ref.at[pl.ds(input_offsets[id], send_sizes[id]), ...]
    # dst = dst_ref.at[pl.ds(output_offsets[id], send_sizes[id]), ...]
    dst = dst_ref.reshape((dst_ref.shape[0], 8, 128)).at[pl.ds(output_offsets[id], send_sizes[id]), ...]
    copy = pltpu.make_async_copy(src, dst, sems.at[0, sem_id, 1])
    return RDMACopy(copy, copy.start, copy.wait)

  dma_copy = make_dma(idx)
  dma_copy.start()

  def make_rdma(other_id, send: bool = True):
    src_id, dst_id = (idx, other_id) if send else (other_id, idx)
    size = lax.select(idx == src_id, send_sizes[dst_id], recv_sizes[src_id])
    src = src_ref.at[pl.ds(input_offsets[dst_id], size), ...]
    dst = dst_ref.at[pl.ds(output_offsets[dst_id], size), ...]
    sem_id, direction_id = lax.rem(idx + other_id, n_devices), (src_id > dst_id).astype(jnp.int32)
    send_sem = sems.at[sem_id, direction_id, 0]
    recv_sem = sems.at[sem_id, direction_id, 1]
    copy = pltpu.make_async_remote_copy(src, dst, send_sem, recv_sem, device_id=dst_id)
    start_fn = copy.start if send else (lambda: None)
    wait_fn = copy.wait_send if send else copy.wait_recv
    return RDMACopy(copy, start_fn, wait_fn)

  # send_rdmas, recv_rdmas = [], []
  # for i in range(1, n_devices):
  #   other_id = jax.lax.rem(idx + i, n_devices)
  #   send_rdmas.append(make_rdma(other_id, send=True))
  #   recv_rdmas.append(make_rdma(other_id, send=False))
  # [rdma.start() for rdma in send_rdmas]
  # [rdma.wait() for rdma in (send_rdmas + recv_rdmas)]
  dma_copy.wait()

@partial(jax.jit, static_argnames=("axis_name",))
def ra2a(src, output, input_offsets, send_sizes, output_offsets, recv_sizes, *, axis_name: str = "x"):
  n_devices = jax.lax.axis_size(axis_name)
  out = pl.pallas_call(
    partial(ra2a_kernel, axis_name=axis_name),
    out_shape=output,
    in_specs=2 * [pl.BlockSpec(memory_space=pltpu.ANY)] + 4 * [pl.BlockSpec(memory_space=pltpu.SMEM)],
    out_specs=pl.BlockSpec(memory_space=pltpu.ANY),
    scratch_shapes=[pltpu.SemaphoreType.DMA((n_devices, 2, 2))],
    input_output_aliases={1: 0},
    interpret=False,
  )(src, output, input_offsets, send_sizes, output_offsets, recv_sizes)
  return out


In [29]:
@partial(jax.jit, static_argnames=("n", "k"))
def generate_data(n, k):
  x = auto_axes(lambda: jnp.tile(jnp.arange(n)[:, None, None], (1, 8, k // 8)), out_sharding=P("x", None, None))()
  idx = random.randint(random.key(0), shape=(n,), minval=0, maxval=DEVICE_NUM)

  @partial(jax.shard_map, in_specs=(P("x", None, None), P(None)), out_specs=(P("x", None, None),) + (P("x"),) * 4)
  def fn(x, idx):
    id = jax.lax.axis_index("x")
    local_idx = jax.lax.dynamic_slice_in_dim(idx, id * x.shape[0], x.shape[0], axis=0)

    sizes = jax.vmap(lambda idx: jnp.bincount(idx, length=DEVICE_NUM))(idx.reshape((DEVICE_NUM, -1)))

    send_sizes = jnp.take_along_axis(sizes, id[None, None], axis=0)[0, :]
    recv_sizes = jnp.take_along_axis(sizes, id[None, None], axis=1)[:, 0]

    input_offsets = jnp.concat([jnp.zeros((1,), send_sizes.dtype), jnp.cumsum(send_sizes)[:-1]])

    output_offsets = jnp.concat([jnp.zeros((1, sizes.shape[0]), sizes.dtype), jnp.cumsum(sizes, 0)])[:-1, :]
    output_offsets = jnp.take_along_axis(output_offsets, id[None, None], axis=0)[0, :]

    x_sort = jnp.take_along_axis(x, jnp.argsort(local_idx)[:, None, None], 0)
    return x_sort, input_offsets, send_sizes, output_offsets, recv_sizes

  return fn(x, idx)


In [30]:
@jax.jit
def transfer(x):
  @partial(jax.shard_map, out_specs=P("x", None))
  def fn(x):
    peer_num = jax.lax.axis_size("x")
    chunk_shape = x.shape[0] // peer_num
    print(f"chunk_shape = {chunk_shape}")
    input_offsets = chunk_shape * jnp.arange(peer_num)
    output_offsets = chunk_shape * jnp.arange(peer_num)
    send_sizes = chunk_shape * jnp.ones(peer_num, "int32")
    recv_sizes = chunk_shape * jnp.ones(peer_num, "int32")
    output = jnp.zeros((x.shape[0], x.shape[1] * x.shape[2]), x.dtype)

    #return jnp.sum(jax.lax.all_to_all(output, "x", 0, 0, tiled=True), 0)[None, :]

    #return jnp.sum(output, 0)[None, :]

    #out = jax.lax.ragged_all_to_all(x, output, input_offsets, send_sizes, output_offsets, recv_sizes, axis_name="x")

    out = ra2a(x, output, input_offsets, send_sizes, output_offsets, recv_sizes, axis_name="x")
    #out = ra2a(x, x, input_offsets, send_sizes, output_offsets, recv_sizes, axis_name="x")
    return jnp.sum(out, 0)[None, :]

  return fn(x)

In [31]:
x, input_offsets, send_sizes, output_offsets, recv_sizes = generate_data(128, 8 * 128)

In [ ]:
transfer(x)

# MoE

In [6]:
@partial(jax.tree_util.register_dataclass, data_fields=[
  "input_offsets", "send_sizes", "output_offsets", "recv_sizes"
], meta_fields=[])
@dataclasses.dataclass
class RA2AMeta:
  input_offsets: jax.Array
  send_sizes: jax.Array
  output_offsets: jax.Array
  recv_sizes: jax.Array

In [34]:
def empty(shape, dtype, out_sharding=None):
  if out_sharding is None:
    out_shape, out_specs = jax.ShapeDtypeStruct(shape, dtype), pl.BlockSpec(memory_space=pltpu.ANY)
    return pl.pallas_call(lambda *args: None, out_shape=out_shape, out_specs=out_specs)()

  assert len(out_sharding) <= len(shape)
  shard_axes = tuple(out_sharding) + (None,) * (len(shape) - len(out_sharding))
  
  @partial(jax.shard_map, out_specs=out_sharding, check_vma=False)
  def _():
    local_shape = [(s // jax.lax.axis_size(a)) if a is not None else s for s, a in zip(shape, shard_axes)]
    out_shape, out_specs = jax.ShapeDtypeStruct(local_shape, dtype), pl.BlockSpec(memory_space=pltpu.ANY)
    return pl.pallas_call(lambda *args: None, out_shape=out_shape, out_specs=out_specs)()
  return _()


In [ ]:
jax.typeof(empty((128, 8, 128), jnp.bfloat16, out_sharding=P("x")))

ShapedArray(bfloat16[128@x,8,128])

In [ ]:
def make_compute_metadata(axis_name, experts_num, safety_factor: int = 2):
  @partial(jax.shard_map, out_specs=P("x", None, None), check_vma=False)
  def compute_metadata(x: jax.Array, all_idxs: jax.Array):
    shard_idx, num_shards = jax.lax.axis_index(axis_name), jax.lax.axis_size(axis_name)
    experts_per_shard = experts_num // num_shards
    experts_per_tok = all_idxs.size // x.shape[0] // num_shards  # because all_idxs is replicated

    if all_idxs.ndim == 1:
      all_idxs = all_idxs.reshape((num_shards, -1))

    # compute the ra2a communication ###################################################################################
    
    all_sizes = jax.vmap(partial(jnp.bincount, length=num_shards))(all_idxs // experts_per_shard)
    all_input_offsets = jnp.cumsum(all_sizes, axis=-1) - all_sizes  # cumsum from 0
    all_output_offsets = jnp.cumsum(all_sizes, axis=0) - all_sizes  # cumsum from 0
    send_sizes, recv_sizes = all_sizes[shard_idx, :], all_sizes[:, shard_idx]
    input_offsets, output_offsets = all_input_offsets[shard_idx, :], all_output_offsets[shard_idx, :]
    preamble = RA2AMeta(input_offsets, send_sizes, output_offsets, recv_sizes)
    inv_input_offsets = all_output_offsets[:, shard_idx]  # we send back chunks starting where we received them
    inv_output_offsets = all_input_offsets[:, shard_idx]  # we write the chunks from where they originally came
    inv_send_sizes, inv_recv_sizes = recv_sizes, send_sizes
    epilogue = RA2AMeta(inv_input_offsets, inv_send_sizes, inv_output_offsets, inv_recv_sizes)

    # compute within expert shard sort; receiving several locally sorted chunks ########################################

    # get gather indices for pre-ra2a by-expert-shard organization
    local_ra2a_sort = jnp.argsort(all_idxs[shard_idx, :])
    local_ra2a_isort = jnp.argsort(local_ra2a_sort)

    all_expert_idxs = jax.lax.all_gather(all_idxs[shard_idx, :][local_ra2a_sort], axis_name)

    local_expert_idxs = empty((x.shape[0] * experts_per_tok * safety_factor,), jnp.int32)
    def update_fn(i, local_expert_idxs):
      update = jnp.roll(all_expert_idxs[i, :], -all_input_offsets[i, shard_idx])
      return jax.lax.dynamic_update_slice_in_dim(local_expert_idxs, update, all_output_offsets[i, shard_idx], 0)
    local_expert_idxs = jax.lax.fori_loop(0, num_shards, update_fn, local_expert_idxs)

    # ra2a_sort = jnp.argsort(all_idxs, axis=-1)
    # all_expert_idxs = jnp.take_along_axis(all_idxs, ra2a_sort, axis=-1)  # expensive
    # all_shard_assignment = all_expert_idxs // experts_per_shard
    # local_mask = ((all_shard_assignment >= shard_idx) & (all_shard_assignment < (shard_idx + 1))).reshape(-1)
    # local_pack_idx = jnp.where(local_mask, size=local_mask.size, fill_value=0)  # expensive
    # local_expert_idxs = all_expert_idxs.reshape(-1)[local_pack_idx]  # expensive
    # local_expert_idxs = local_expert_idxs[:x.shape[0] * experts_per_tok * safety_factor]
    # mask = jnp.arange(local_expert_idxs.size) < jnp.sum(recv_sizes)
    # jax.debug.print("diff = {}", jnp.sum(jnp.abs(jnp.where(mask, local_expert_idxs - local_expert_idxs_, 0))))

    local_pack_mask = jnp.arange(local_expert_idxs.size) < jnp.sum(recv_sizes)
    local_sort = jnp.argsort(jnp.where(local_pack_mask, local_expert_idxs, 2 ** 30))
    local_isort = jnp.argsort(local_sort)
    local_group_sizes = jnp.bincount(
      jnp.where(local_pack_mask, local_expert_idxs - shard_idx * experts_per_shard, 2 ** 30), length=experts_per_shard
    )

    # perform the actual communication and computation #################################################################

    # step 1: gather local tokens for every expert per token
    x_sort = x[local_ra2a_sort // experts_per_tok, ...]
    
    # step 2: communicate expert-gathered-tokens to their corresponding expert shards
    #buffer = jnp.empty((x.shape[0] * experts_per_tok * safety_factor,) + x.shape[1:], dtype=x.dtype)
    buffer = empty((x.shape[0] * experts_per_tok * safety_factor,) + x.shape[1:], dtype=x.dtype)
    y = jax.lax.ragged_all_to_all(x_sort, buffer, *dataclasses.astuple(preamble), axis_name=axis_name)
    
    # step 3: gather tokens locally so they're expert-contiguous
    y = y[local_sort, ...]
    
    # step 4: perform gmm computation
    pass
  
    # step 5: unpermute tokens locally to organize them into chunks in which they arrived
    y = y[local_isort, ...]
    
    # step 6: communincate the chunks back to their origins
    #out = jnp.empty((x.shape[0] * experts_per_tok,) + x.shape[1:], dtype=x.dtype)
    out = empty((x.shape[0] * experts_per_tok,) + x.shape[1:], dtype=x.dtype)
    x_sort = jax.lax.ragged_all_to_all(y, out, *dataclasses.astuple(epilogue), axis_name=axis_name)
    
    # step 7: gather so each token repeats are next to each other
    x = x_sort[local_ra2a_isort, ...].reshape((x.shape[0], experts_per_tok) + x.shape[1:])
    
    # step 8: weigh by expert weights
    pass

    return x

  return compute_metadata

  # 1. gather to separate destination shards
  # 2. ra2a
  # 3. gather to rearrange on local shard

In [87]:
keys = iter(random.split(random.key(0), 1024))
x = random.normal(next(keys), (8 * 4096 * 4, 56, 128), jnp.bfloat16, out_sharding=P("x", None, None))
all_expert_idxs = random.randint(next(keys), x.shape[0] * 2, 0, 256, out_sharding=P())

In [88]:
moe = jax.jit(make_compute_metadata("x", 256, 3))
y = moe(x, all_expert_idxs)

In [89]:
with jax.profiler.trace("/tmp/custom_moe"):
  for _ in range(3):
    y = jax.block_until_ready(moe(x, all_expert_idxs))

In [83]:
np.array(y)[:5, :, 0, 0]

array([[1.15625, 1.15625],
       [0.0634766, 0.0634766],
       [0.202148, 0.202148],
       [0.494141, 0.494141],
       [2.20312, 2.20312]], dtype=bfloat16)

In [84]:
np.array(x)[:5, 0, 0]

array([1.15625, 0.0634766, 0.202148, 0.494141, 2.20312], dtype=bfloat16)

In [100]:
jnp.sum(jnp.sum(jnp.abs(y - x), (-1, -2)))

Array(0, dtype=bfloat16)

In [82]:
np.array(y)[-128:, 0, 0]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=bfloat16)

In [85]:
np.array(y).reshape((4, -1, 8, 128))[:, :, 0, 0]

array([[1.15625, -0.910156, 1.78125, ..., 0, 0, 0],
       [-0.769531, 0.808594, 1.54688, ..., 0, 0, 0],
       [0.283203, -0.439453, -1.46094, ..., 0, 0, 0],
       [-1.51562, -0.439453, 1.32812, ..., 0, 0, 0]],
      shape=(4, 1024), dtype=bfloat16)

In [25]:
r = random.randint(random.key(0), (1024,), 0, 1000)

In [29]:
jnp.all((jnp.cumsum(r) - r) == jnp.concatenate([jnp.zeros((1,), "int32"), jnp.cumsum(r[:-1])]))

Array(True, dtype=bool)

In [24]:
a = jnp.array([5, 12])
jnp.cumsum(a) - a

Array([0, 5], dtype=int32)

In [17]:
m = jnp.array([0, 0, 1, 2, 0, 5, 0, 0])

In [19]:
m[jnp.where(m, size=m.size, fill_value=0)]

Array([1, 2, 5, 0, 0, 0, 0, 0], dtype=int32)

In [16]:
jnp.bincount(jnp.array([1, 2, 3, 4, 4, 4, 4]), length=2)

Array([0, 1], dtype=int32)

In [ ]:
@partial(jax.jit, static_argnames=("n", "k"))
def generate_data(n, k):
  x = auto_axes(lambda: jnp.tile(jnp.arange(n)[:, None, None], (1, 8, k // 8)), out_sharding=P("x", None, None))()
  idx = random.randint(random.key(0), shape=(n,), minval=0, maxval=DEVICE_NUM)

  @partial(jax.shard_map, in_specs=(P("x", None, None), P(None)), out_specs=(P("x", None, None),) + (P("x"),) * 4)
  def fn(x, idx):
    id = jax.lax.axis_index("x")
    local_idx = jax.lax.dynamic_slice_in_dim(idx, id * x.shape[0], x.shape[0], axis=0)

    sizes = jax.vmap(lambda idx: jnp.bincount(idx, length=DEVICE_NUM))(idx.reshape((DEVICE_NUM, -1)))

    send_sizes = jnp.take_along_axis(sizes, id[None, None], axis=0)[0, :]
    recv_sizes = jnp.take_along_axis(sizes, id[None, None], axis=1)[:, 0]

    input_offsets = jnp.concat([jnp.zeros((1,), send_sizes.dtype), jnp.cumsum(send_sizes)[:-1]])

    output_offsets = jnp.concat([jnp.zeros((1, sizes.shape[0]), sizes.dtype), jnp.cumsum(sizes, 0)])[:-1, :]
    output_offsets = jnp.take_along_axis(output_offsets, id[None, None], axis=0)[0, :]

    x_sort = jnp.take_along_axis(x, jnp.argsort(local_idx)[:, None, None], 0)
    return x_sort, input_offsets, send_sizes, output_offsets, recv_sizes

  return fn(x, idx)


In [14]:
x, input_offsets, send_sizes, output_offsets, recv_sizes = generate_data(128, 8 * 128)

In [36]:
@jax.jit
@partial(jax.shard_map, out_specs=P("x", None, None), check_vma=False)
def test_kernel_async(x, input_offsets, send_sizes, output_offsets, recv_sizes):
  output = jnp.zeros((2 * x.shape[0],) + x.shape[1:], x.dtype)
  # lax_ra2a = partial(jax.lax.ragged_all_to_all, axis_name="x")
  start, wait = make_ra2a(axis_name="x")
  #lax_ra2a = partial(ra2a, axis_name="x")
  #return lax_ra2a(x, output, input_offsets, send_sizes, output_offsets, recv_sizes)
  with jax.named_scope("start"):
    out, future = start(x, output, input_offsets, send_sizes, output_offsets, recv_sizes)
  with jax.named_scope("wait"):
    out = wait(x, out, input_offsets, send_sizes, output_offsets, recv_sizes, future)
  with jax.named_scope("jax.lax.ragged_all_to_all"):
    out2 = jax.lax.ragged_all_to_all(x, out, input_offsets, send_sizes, output_offsets, recv_sizes, axis_name="x")
  return out, out2

In [37]:
x4 = jax.block_until_ready(test_kernel_async(x, input_offsets, send_sizes, output_offsets, recv_sizes))

ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?


In [38]:
x, input_offsets, send_sizes, output_offsets, recv_sizes = generate_data(1024 ** 2, 8 * 128)
x4 = jax.block_until_ready(test_kernel_async(x, input_offsets, send_sizes, output_offsets, recv_sizes))
with jax.profiler.trace("/tmp/async_ra2a"):
  for _ in range(3):
    x4 = jax.block_until_ready(test_kernel_async(x, input_offsets, send_sizes, output_offsets, recv_sizes))

In [19]:
@jax.jit
@partial(jax.shard_map, out_specs=P("x", None, None))
def test_xla(x, input_offsets, send_sizes, output_offsets, recv_sizes):
  output = jnp.zeros((2 * x.shape[0],) + x.shape[1:], x.dtype)
  lax_ra2a = partial(jax.lax.ragged_all_to_all, axis_name="x")
  # lax_ra2a = partial(ra2a, axis_name="x")
  return lax_ra2a(x, output, input_offsets, send_sizes, output_offsets, recv_sizes)

@jax.jit
@partial(jax.shard_map, out_specs=P("x", None, None), check_vma=False)
def test_kernel(x, input_offsets, send_sizes, output_offsets, recv_sizes):
  output = jnp.zeros((2 * x.shape[0],) + x.shape[1:], x.dtype)
  # lax_ra2a = partial(jax.lax.ragged_all_to_all, axis_name="x")
  lax_ra2a = partial(ra2a, axis_name="x")
  return lax_ra2a(x, output, input_offsets, send_sizes, output_offsets, recv_sizes)

In [20]:
x2 = jax.block_until_ready(test_kernel(x, input_offsets, send_sizes, output_offsets, recv_sizes))
#x2 = test_kernel(x, input_offsets, send_sizes, output_offsets, recv_sizes)

In [22]:
x1 = test_xla(x, input_offsets, send_sizes, output_offsets, recv_sizes)
#x2 = test_kernel(x, input_offsets, send_sizes, output_offsets, recv_sizes)
x2 = test_kernel(x, input_offsets, send_sizes, output_offsets, recv_sizes)
x3 = test_xla(x, input_offsets, send_sizes, output_offsets, recv_sizes)
with np.printoptions(linewidth=700):
  print(jnp.sum(jnp.abs(x1 - x2), axis=(-1, -2)).reshape((4, -1)))
  print("---")
  print(jnp.sum(jnp.abs(x1 - x3), axis=(-1, -2)).reshape((4, -1)))
  print("---")
  print(jnp.sum(jnp.abs(x1 - x4), axis=(-1, -2)).reshape((4, -1)))

[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]
---
[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 

In [30]:
print(np.array(send_sizes).reshape((-1, DEVICE_NUM)))
print("--------")
print(np.array(recv_sizes).reshape((-1, DEVICE_NUM)))
print(np.array(output_offsets).reshape((DEVICE_NUM, -1)))
print(np.array(input_offsets).reshape((-1, DEVICE_NUM)))
print("--------")
print(np.array(output_offsets).reshape((-1, DEVICE_NUM)))
print(np.array(recv_sizes).reshape((DEVICE_NUM, -1)))
rs = np.array(recv_sizes).reshape((DEVICE_NUM, -1))
np.concat([np.zeros((1, rs.shape[0]), rs.dtype), np.cumsum(rs, axis=0)])[:-1, :]

[[11  8  4  9]
 [ 5  8  9 10]
 [10  8  9  5]
 [ 6  8  5 13]]
--------
[[11  5 10  6]
 [ 8  8  8  8]
 [ 4  9  9  5]
 [ 9 10  5 13]]
[[ 0  0  0  0]
 [11  8  4  9]
 [16 16 13 19]
 [26 24 22 24]]
[[ 0 11 19 23]
 [ 0  5 13 22]
 [ 0 10 18 27]
 [ 0  6 14 19]]
--------
[[ 0  0  0  0]
 [11  8  4  9]
 [16 16 13 19]
 [26 24 22 24]]
[[11  5 10  6]
 [ 8  8  8  8]
 [ 4  9  9  5]
 [ 9 10  5 13]]


array([[ 0,  0,  0,  0],
       [11,  5, 10,  6],
       [19, 13, 18, 14],
       [23, 22, 27, 19]])

In [275]:
np.sum(np.array(send_sizes).reshape((DEVICE_NUM, -1)), 0)

array([32, 32, 27, 37])

In [285]:
np.array(x1)[-33:, 0, 0]

array([109, 110, 113, 116, 119, 123,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0], dtype=int32)

In [279]:
print(np.array(send_sizes).reshape((DEVICE_NUM, -1)))
print("---")
print(np.array(output_offsets).reshape((DEVICE_NUM, -1)))

[[11  8  4  9]
 [ 5  8  9 10]
 [10  8  9  5]
 [ 6  8  5 13]]
---
[[ 0  0  0  0]
 [11  8  4  9]
 [16 16 13 19]
 [26 24 22 24]]


In [273]:
np.cumsum(np.array(recv_sizes).reshape((-1, DEVICE_NUM)), axis=-1)

array([[11, 16, 26, 32],
       [ 8, 16, 24, 32],
       [ 4, 13, 22, 27],
       [ 9, 19, 24, 37]])

In [274]:
np.array(send_sizes).reshape((DEVICE_NUM, -1))

array([[11,  8,  4,  9],
       [ 5,  8,  9, 10],
       [10,  8,  9,  5],
       [ 6,  8,  5, 13]], dtype=int32)

In [243]:
np.array(recv_sizes).reshape((DEVICE_NUM, -1))

array([[11,  5, 10,  6],
       [ 8,  8,  8,  8],
       [ 4,  9,  9,  5],
       [ 9, 10,  5, 13]], dtype=int32)

In [175]:
print(np.cumsum(np.array(send_sizes).reshape((-1, DEVICE_NUM)), axis=-1))
print("---")
print(np.cumsum(np.array(send_sizes).reshape((-1, DEVICE_NUM)), axis=0))

[[11 19 23 32]
 [ 5 13 22 32]
 [10 18 27 32]
 [ 6 14 19 32]]
---
[[11  8  4  9]
 [16 16 13 19]
 [26 24 22 24]
 [32 32 27 37]]


In [ ]:
@jax.jit
def transfer(x):
  @partial(jax.shard_map, out_specs=P("x", None))
  def fn(x):
    peer_num = jax.lax.axis_size("x")
    chunk_shape = x.shape[0] // peer_num
    print(f"chunk_shape = {chunk_shape}")
    input_offsets = chunk_shape * jnp.arange(peer_num)
    output_offsets = chunk_shape * jnp.arange(peer_num)
    send_sizes = chunk_shape * jnp.ones(peer_num, "int32")
    recv_sizes = chunk_shape * jnp.ones(peer_num, "int32")
    output = jnp.zeros_like(x)

    #return jnp.sum(jax.lax.all_to_all(output, "x", 0, 0, tiled=True), 0)[None, :]

    #return jnp.sum(output, 0)[None, :]

    #out = jax.lax.ragged_all_to_all(x, output, input_offsets, send_sizes, output_offsets, recv_sizes, axis_name="x")

    out = ra2a(x, output, input_offsets, send_sizes, output_offsets, recv_sizes, axis_name="x")
    #out = ra2a(x, x, input_offsets, send_sizes, output_offsets, recv_sizes, axis_name="x")
    return jnp.sum(out, 0)[None, :]

  return fn(x)

In [7]:
#jax.config.update("jax_traceback_filtering", "off")
# y = transfer(x)
with jax.profiler.trace("/tmp/ra2a_custom"):
  for _ in range(3):
    y_ = jax.block_until_ready(transfer(x_))

chunk_shape = 65536


In [ ]:
DEVICE_NUM = jax.device_count()
mesh = jax.make_mesh((DEVICE_NUM,), ("x",), axis_types=jax.sharding.AxisType.Explicit)
print(mesh)
jax.sharding.set_mesh(mesh)

In [ ]:
x = jnp.tile(jnp.repeat(1 + jnp.arange(DEVICE_NUM), 128, axis=0)[..., None], (1, 128))
x = jax.device_put(x, P("x", None))

In [ ]:
x.format

In [ ]:
input_offsets = 0 * jnp.ones(DEVICE_NUM, "int32")
send_sizes = 1 * jnp.ones(DEVICE_NUM, "int32")
output_offsets = 1 * jnp.ones(DEVICE_NUM, "int32")
#recv_sizes = 128 * jnp.ones(DEVICE_NUM, "int32")
recv_sizes = send_sizes

In [ ]:
@jax.jit
@partial(jax.shard_map, out_specs=P("x", None))
def fn(x, input_offsets, send_sizes, output_offsets, recv_sizes):
  return ra2a(x, x, input_offsets, send_sizes, output_offsets, recv_sizes)

In [ ]:
x.shape

In [ ]:
y = jax.block_until_ready(fn(x, input_offsets, send_sizes, output_offsets, recv_sizes))

In [ ]:
y

In [ ]:
i = 1
#y.at[i * 128:(i + 1) * 128, :].get(out_sharding=P())[:, 0]
y.at[128:128+128, :].get(out_sharding=P())[:33, 0]

In [ ]:
x.at[i * 128:(i + 1) * 128, :].get(out_sharding=P())[:, 0]